In [ ]:
! mkdir -p /root/.config/kaggle && mv /root/.config/kaggle.json /root/.config/kaggle/kaggle.json
# Make sure your kaagle api is available in here
# /root/.config/kaggle/kaggle.json
!kaggle competitions download -c rsna-2024-lumbar-spine-degenerative-classification
# 2025-05-7
! ls
! unzip -q /content/drive/MyDrive/Datasets/Cats\&Dogs.zip
img_path = "/content/dataset/training_set/cats/cat.1.jpg"
import cv2
import PIL

img = cv2.imread(img_path)
img = PIL.Image.open(img_path)
type(img)
# train_set = torchvision.datasets.CIFAR10('./data', train=True)
# train_loader = DataLoader(train_set, batch_size=16)
# for img, label in train_loader:
#     pred = model(img)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# class CatsDogs(Dataset):
#     def __init__(self):
#         pass
#     def __len__(self):
#         pass
#     def __getitem__(self, idx):
#         pass
img1_path = '/content/dataset/training_set/cats/cat.1.jpg'
img2_path = '/content/dataset/training_set/cats/cat.2.jpg'
img100_path = '/content/dataset/training_set/cats/cat.100.jpg'
root_path = "/content/dataset/training_set/"
import os

os.listdir(root_path)
os.listdir('/content/drive/')
os.listdir("/content/dataset/training_set/dogs/")
os.listdir(root_path)
os.path.join(root_path, 'dogs')
root_path
os.path.join(root_path, 'cats')
os.listdir("/content/dataset/training_set/cats")


class CatsDogs(Dataset):
    def __init__(self, root_path, transform):
        self.root_path = root_path
        self.imgs_path = []
        self.labels = []
        self.transform = transform

        for id, class_name in enumerate(os.listdir(self.root_path)):
            class_path = os.path.join(self.root_path, class_name)
            for img_name in os.listdir(class_path):
                img_path = os.path.join(self.root_path, class_name, img_name)
                # cv2.imread(img_path)
                self.imgs_path.append(img_path)
                self.labels.append(id)

    def __len__(self):
        return len(self.imgs_path)

    def __getitem__(self, idx):
        img = PIL.Image.open(self.imgs_path[idx])
        label = self.labels[idx]

        img = self.transform(img)

        return img, label


# train_set = CatsDogs('./data')
# train_loader = DataLoader(train_set, batch_size=16)
# for img, label in train_loader:
#     pred = model(img)
import torchvision

transfrom = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor()
])

train_data = CatsDogs(root_path='/content/dataset/training_set', transform=transfrom)
test_data = CatsDogs(root_path='/content/dataset/test_set', transform=transfrom)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
len(train_data), len(test_data)
for imgs, labels in train_loader:
    break
import matplotlib.pyplot as plt

plt.imshow(imgs[7].permute(1, 2, 0).numpy())
from torchvision.models import resnet18

model = resnet18()
model
model.fc
model.fc = nn.Linear(in_features=model.fc.in_features, out_features=2)
model.fc
from sklearn.metrics import accuracy_score


def train(loader, model, criterion, optimizer, device):
    loss_per_epoch = []
    acc_per_epoch = []
    for data, label in loader:
        optimizer.zero_grad()

        data = data.to(device)
        label = label.to(device)

        pred = model(data)
        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()

        acc = accuracy_score(label.detach().cpu(), pred.argmax(dim=1).detach().cpu())
        acc_per_epoch.append(acc)
        loss_per_epoch.append(loss.item())

    return torch.mean(torch.tensor(loss_per_epoch)), torch.mean(torch.tensor(acc_per_epoch))


def test(loader, model, criterion, device):
    loss_per_epoch = []
    acc_per_epoch = []
    with torch.no_grad():
        for data, label in loader:
            data = data.to(device)
            label = label.to(device)

            pred = model(data)
            loss = criterion(pred, label)
            acc = accuracy_score(label.detach().cpu(), pred.argmax(dim=1).detach().cpu())
            acc_per_epoch.append(acc)
            loss_per_epoch.append(loss.item())

    return torch.mean(torch.tensor(loss_per_epoch)), torch.mean(torch.tensor(acc_per_epoch))


import PIL

device = 'cuda'
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()
model = model.to(device)

train_loss_per_epoch = []
train_acc_per_epoch = []
test_loss_per_epoch = []
test_acc_per_epoch = []
for epoch in range(10):
    train_loss, train_acc = train(train_loader, model, criterion, optimizer, device)
    test_loss, test_acc = test(test_loader, model, criterion, device)
    train_loss_per_epoch.append(train_loss)
    train_acc_per_epoch.append(train_acc)
    test_loss_per_epoch.append(test_loss)
    test_acc_per_epoch.append(test_acc)
    print(
        f"Epoch {epoch + 1}/10: Train loss: {train_loss}, Train acc: {train_acc}, Test loss: {test_loss}, Test acc: {test_acc}")
